In [25]:
"""
Logistic Regression Regularization Demo
========================================
Interactive visualization showing how the regularization parameter λ (lambda)
and polynomial degree affect the decision boundary in logistic regression.

- Move the λ slider LEFT  → less regularization → model overfits (wiggly boundary)
- Move the λ slider RIGHT → more regularization → model underfits (too simple)
- Increase polynomial degree → more complex model → easier to overfit
- The sweet spot in the middle gives the best test accuracy

Requirements: scikit-learn, matplotlib, ipywidgets, numpy
Run in VS Code with the Jupyter extension (.ipynb) or as a notebook cell.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import ipywidgets as widgets
from IPython.display import display, clear_output



In [26]:
# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42

# ── Generate dataset ─────────────────────────────────────────────────────────
X, y = make_moons(n_samples=200, noise=0.25, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

# ── Colour palette ────────────────────────────────────────────────────────────
CLASS_COLORS   = ["#F15B97", "#F172F7"]
REGION_COLORS  = ["#F12A7D", "#BA41ED"]
BOUNDARY_COLOR = "#2D2D2D"

cmap_points  = ListedColormap(CLASS_COLORS)
cmap_regions = ListedColormap(REGION_COLORS)

# ── Mesh grid for decision boundary ──────────────────────────────────────────
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))



In [27]:
# ── Build and fit model ───────────────────────────────────────────────────────
def build_model(lam: float, degree: int) -> Pipeline:
    """Return a fitted pipeline for the given λ and polynomial degree."""
    C = 1.0 / lam  # sklearn uses C = 1/λ
    pipe = Pipeline([
        ("poly",   PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(C=C, max_iter=10_000, random_state=RANDOM_STATE)),
    ])
    pipe.fit(X_train, y_train)
    return pipe



In [28]:
# ── Lambda label helper ───────────────────────────────────────────────────────
def lambda_label(lam: float) -> str:
    if lam < 0.01:
        return f"λ = {lam:.4f}  (very low — risk of overfitting)"
    elif lam < 0.5:
        return f"λ = {lam:.3f}  (low — some overfitting)"
    elif lam < 5:
        return f"λ = {lam:.2f}  (balanced — good generalisation)"
    elif lam < 50:
        return f"λ = {lam:.1f}  (high — starting to underfit)"
    else:
        return f"λ = {lam:.1f}  (very high — underfitting)"



In [29]:
# ── Accuracy bar helper ───────────────────────────────────────────────────────
def draw_accuracy_bars(ax, train_acc: float, test_acc: float) -> None:
    bars = ax.barh(
        ["Train accuracy", "Test accuracy"],
        [train_acc * 100, test_acc * 100],
        color=["#EA6DBC", "#DB73F0"],
        height=0.4,
        edgecolor="white",
        linewidth=1.5,
    )
    ax.set_xlim(0, 105)
    ax.set_xlabel("Accuracy (%)", fontsize=10)
    ax.set_title("Model accuracy", fontsize=11, fontweight="bold", pad=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=10)

    for bar, val in zip(bars, [train_acc, test_acc]):
        ax.text(
            bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{val*100:.1f}%",
            va="center", ha="left", fontsize=10, fontweight="bold",
        )

    gap = abs(train_acc - test_acc) * 100
    ax.text(
        52, -0.75,
        f"Train–test gap: {gap:.1f}%",
        ha="center", fontsize=9,
        color="#555" if gap < 5 else "#F174C7",
        style="italic",
    )



In [30]:
# ── Widget setup ──────────────────────────────────────────────────────────────
lambda_slider = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-3,    # 10^-3 = 0.001
    max=3,     # 10^3  = 1000
    step=0.05,
    description="λ (lambda)",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
    style={"description_width": "90px"},
)
 
degree_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    step=1,
    description="Poly degree",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
    style={"description_width": "90px"},
)
 
hint = widgets.HTML(
    value=(
        "<div style='font-size:12px; color:#555; margin-top:6px;'>"
        "◀ &nbsp;<b>λ left</b> = overfitting &nbsp;|&nbsp; "
        "<b>λ right</b> = underfitting &nbsp;▶"
        "&nbsp;&nbsp;&nbsp;|&nbsp;&nbsp;&nbsp;"
        "<b>Degree</b>: higher = more complex model, easier to overfit"
        "</div>"
    )
)
 
# Output widget captures the matplotlib figure
plot_output = widgets.Output()


In [31]:
# ── Update function ───────────────────────────────────────────────────────────
def on_change(change):
    with plot_output:
        clear_output(wait=True)
 
        lam    = lambda_slider.value
        degree = degree_slider.value
 
        pipe = build_model(lam, degree)
        train_acc = pipe.score(X_train, y_train)
        test_acc  = pipe.score(X_test,  y_test)
 
        Z = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
 
        fig, (ax_boundary, ax_accuracy) = plt.subplots(
            1, 2,
            figsize=(13, 5.5),
            gridspec_kw={"width_ratios": [1.6, 1], "wspace": 0.35},
        )
        fig.patch.set_facecolor("#F8F9FA")
        fig.subplots_adjust(left=0.06, right=0.97, top=0.88, bottom=0.13)
        fig.suptitle(
            f"Logistic Regression — Degree {degree} Polynomial  |  {lambda_label(lam)}",
            fontsize=12, fontweight="bold", y=0.97, color="#1A1A2E",
        )
 
        # ── Decision boundary panel ───────────────────────────────────────
        ax_boundary.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_regions)
        ax_boundary.contour(xx, yy, Z, levels=[0.5], colors=BOUNDARY_COLOR,
                            linewidths=2)
        ax_boundary.scatter(
            X_train[:, 0], X_train[:, 1], c=y_train,
            cmap=cmap_points, edgecolors="white", linewidths=0.6,
            s=60, zorder=3, label="Train",
        )
        ax_boundary.scatter(
            X_test[:, 0], X_test[:, 1], c=y_test,
            cmap=cmap_points, edgecolors="black", linewidths=0.8,
            s=80, marker="D", zorder=4, alpha=0.85, label="Test",
        )
        ax_boundary.set_xlim(x_min, x_max)
        ax_boundary.set_ylim(y_min, y_max)
        ax_boundary.set_xlabel("Feature 1", fontsize=10)
        ax_boundary.set_ylabel("Feature 2", fontsize=10)
        ax_boundary.set_title("Decision boundary", fontsize=11,
                               fontweight="bold", pad=8)
        ax_boundary.legend(loc="upper right", fontsize=9,
                           framealpha=0.85, edgecolor="#ccc")
        ax_boundary.spines[["top", "right"]].set_visible(False)
 
        # ── Accuracy panel ────────────────────────────────────────────────
        draw_accuracy_bars(ax_accuracy, train_acc, test_acc)
 
        plt.show()
 

In [32]:
# ── Wire up and display ───────────────────────────────────────────────────────
lambda_slider.observe(on_change, names="value")
degree_slider.observe(on_change, names="value")
 
ui = widgets.VBox(
    [lambda_slider, degree_slider, hint, plot_output],
    layout=widgets.Layout(margin="8px 0 0 10px"),
)
 
display(ui)
 
# Draw initial chart
on_change(None)